# NB07: Bias correction and uncertainty (Phase 7R)

Four-track bias correction A/B test with both train and test RMSE, plus a
QRF leakage A/B test that contrasts coverage measured on the training set
against out-of-fold coverage. All feature construction and model loading is
driven by the Phase 3R winning configuration JSON; nothing is hard-coded.

## Purpose / Inputs / Outputs / Canonical decisions

**Purpose:** Bias correction and uncertainty quantification. Four-track A/B test of piecewise linear residual corrections (plus paired-bootstrap null-result check), QRF coverage leakage A/B, and multi-alpha split-conformal calibration (primary: 90% nominal).

**Inputs:** canonical opx-liq RF models, `nb03_multi_seed_results.csv` (for RF hyperparameters), split indices.

**Outputs:** `results/nb07_ab_test_report.csv`, `results/nb07_piecewise_params.json`, `results/nb07_qrf_ab_coverage.csv`, `results/nb07_test_predictions.csv`, `results/nb07_conformal_multi_alpha.csv`, `results/nb07_conformal_qhat.json`, `results/nb07_bias_correction_null_result.csv`, `models/model_QRF_*_opx_liq.joblib`.

**Canonical decisions:** Split-conformal is the **primary** uncertainty estimator for NB08 / NB10 / manuscript. The legacy single-alpha JSON format is preserved (populated from alpha=0.10) for back-compat. Piecewise bias correction is reported as supplementary because the Delta-RMSE 95% CI contains zero.


In [1]:
import sys
import json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from config import (
    ROOT, DATA_RAW, DATA_EXTERNAL, DATA_PROC, DATA_SPLITS, DATA_NATURAL,
    MODELS, FIGURES, RESULTS, LOGS,
    EXPETDB, LEPR_XLSX, LIN2023_NATURAL,
    FE3_FET_RATIO, KD_FEMG_MIN, KD_FEMG_MAX, WO_MAX_MOL_PCT,
    P_CEILING_KBAR, CATION_SUM_MIN, CATION_SUM_MAX,
    OXIDE_TOTAL_MIN, OXIDE_TOTAL_MAX,
    SEED_SPLIT, SEED_MODEL, SEED_NOISE_AUG, SEED_KMEANS,
    OPX_RAW_OXIDES, OPX_FULL_OXIDES, LIQ_OXIDES,
    CONFORMAL_ALPHA,
)
from src.plot_style import load_winning_config, canonical_model_filename
import warnings
warnings.filterwarnings('ignore')

import ast
import numpy as np
import pandas as pd
import joblib
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from quantile_forest import RandomForestQuantileRegressor

In [2]:
# Canonical features and prediction helpers from src/ (one source of truth).
from src.features import (
    build_feature_matrix,
    make_raw_features,
    make_alr_features,
    make_pwlr_features,
    augment_dataframe,
)
from src.models import predict_median, predict_iqr


In [3]:
# Phase 7R setup: load data, splits, canonical winning feature set, and RF models
config_3r = load_winning_config(RESULTS)
WIN_FEAT = config_3r['global_feature_set']
print(f'Phase 3R global winning feature set: {WIN_FEAT}')

df_liq = pd.read_parquet(DATA_PROC / 'opx_clean_opx_liq.parquet')
idx_tr = np.load(DATA_SPLITS / 'train_indices_opx_liq.npy')
idx_te = np.load(DATA_SPLITS / 'test_indices_opx_liq.npy')

df_train = df_liq.loc[idx_tr].copy()
df_test = df_liq.loc[idx_te].copy()

def parse_params(s):
    import ast, json
    if isinstance(s, dict): return s
    try: return ast.literal_eval(s)
    except:
        try: return json.loads(s)
        except: return {}

# Reconstruct RF opx-liq hyperparameters for the winning feature set from the
# multi-seed results table (seed-42 rows are the canonical reference).
multi = pd.read_csv(RESULTS / 'nb03_multi_seed_results.csv')
rf_liq = multi[(multi.track == 'opx_liq') &
               (multi.model_name == 'RF') &
               (multi.feature_set == WIN_FEAT) &
               (multi.split_seed == 42)]

params_T = parse_params(rf_liq[rf_liq.target == 'T_C'].iloc[0]['best_params'])
params_P = parse_params(rf_liq[rf_liq.target == 'P_kbar'].iloc[0]['best_params'])

# Load canonical RF models (same feature set for T and P under dynamic selection)
model_T = joblib.load(MODELS / canonical_model_filename('RF', 'T_C', 'opx_liq', RESULTS))
model_P = joblib.load(MODELS / canonical_model_filename('RF', 'P_kbar', 'opx_liq', RESULTS))

# Build feature matrices using the standalone function
X_train, feat_names = build_feature_matrix(df_train, WIN_FEAT, use_liq=True)
X_test, _ = build_feature_matrix(df_test, WIN_FEAT, use_liq=True)

y_train_T = df_train['T_C'].values
y_train_P = df_train['P_kbar'].values
y_test_T = df_test['T_C'].values
y_test_P = df_test['P_kbar'].values

# Groups for StratifiedGroupKFold (used in OOF tracks)
groups_train = df_train['Citation'].astype(str).values
print(f'train n={len(df_train)} test n={len(df_test)} features={len(feat_names)}')

Phase 3R global winning feature set: pwlr


train n=426 test n=174 features=72


In [4]:
# Phase 7R.1: Out-of-fold RF predictions on the training set (10-fold SGKF)
# These OOF predictions are used to fit the unbiased bias-correction tracks
# and to measure OOF coverage for the QRF leakage test.

N_FOLDS = 10

def stratify_labels(y):
    return pd.qcut(y, q=5, labels=False, duplicates='drop')


def oof_rf(X, y, groups, params, seed):
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof = np.zeros_like(y, dtype=float)
    y_strat = stratify_labels(y)
    for fold, (tr, va) in enumerate(sgkf.split(X, y_strat, groups)):
        rf = RandomForestRegressor(**params, random_state=seed, n_jobs=-1)
        rf.fit(X[tr], y[tr])
        oof[va] = predict_median(rf, X[va])
    return oof


def oof_qrf(X, y, groups, params, seed, quantiles=(0.16, 0.5, 0.84)):
    sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof_lo = np.zeros_like(y, dtype=float)
    oof_md = np.zeros_like(y, dtype=float)
    oof_hi = np.zeros_like(y, dtype=float)
    y_strat = stratify_labels(y)
    for fold, (tr, va) in enumerate(sgkf.split(X, y_strat, groups)):
        qrf = RandomForestQuantileRegressor(**params, random_state=seed, n_jobs=-1)
        qrf.fit(X[tr], y[tr])
        q = qrf.predict(X[va], quantiles=list(quantiles))
        oof_lo[va] = q[:, 0]
        oof_md[va] = q[:, 1]
        oof_hi[va] = q[:, 2]
    return oof_lo, oof_md, oof_hi


oof_T = oof_rf(X_train, y_train_T, groups_train, params_T, SEED_MODEL)
oof_P = oof_rf(X_train, y_train_P, groups_train, params_P, SEED_MODEL)

# Canonical-model predictions on train and test (used for Track 1 and as the
# "leak" comparison point in Track 2).
train_pred_T_raw = predict_median(model_T, X_train)
train_pred_P_raw = predict_median(model_P, X_train)
test_pred_T_raw = predict_median(model_T, X_test)
test_pred_P_raw = predict_median(model_P, X_test)
print(f'OOF done. OOF T RMSE = {np.sqrt(mean_squared_error(y_train_T, oof_T)):.2f}, '
      f'OOF P RMSE = {np.sqrt(mean_squared_error(y_train_P, oof_P)):.2f}')

OOF done. OOF T RMSE = 73.37, OOF P RMSE = 5.58


In [5]:
# Phase 7R.2: Piecewise correction (quintile bins of predicted value)

def fit_piecewise_correction(pred, truth, n_bins=5):
    """Fit a piecewise linear correction on (pred, truth-pred) in quintile bins
    of the prediction. Returns bin edges and per-bin (slope, intercept). Edges
    are taken from the OOF distribution so the correction generalizes."""
    edges = np.quantile(pred, np.linspace(0, 1, n_bins + 1))
    edges[0] -= 1e-6
    edges[-1] += 1e-6
    bin_params = []
    for i in range(n_bins):
        mask = (pred >= edges[i]) & (pred < edges[i+1])
        if mask.sum() >= 10:
            lr = LinearRegression().fit(pred[mask].reshape(-1, 1), (truth - pred)[mask])
            slope = float(lr.coef_[0])
            intercept = float(lr.intercept_)
        else:
            slope, intercept = 0.0, 0.0
        bin_params.append({'bin': i, 'lo': float(edges[i]), 'hi': float(edges[i+1]),
                           'slope': slope, 'intercept': intercept, 'n': int(mask.sum())})
    return {'edges': edges.tolist(), 'bins': bin_params}


def apply_piecewise_correction(pred, pw):
    out = pred.copy()
    edges = np.asarray(pw['edges'])
    for b in pw['bins']:
        mask = (pred >= b['lo']) & (pred < b['hi'])
        out[mask] = pred[mask] + b['slope'] * pred[mask] + b['intercept']
    # Handle any predictions outside fitted edges (clip to nearest bin)
    below = pred < edges[0]
    above = pred >= edges[-1]
    if below.any():
        b0 = pw['bins'][0]
        out[below] = pred[below] + b0['slope'] * pred[below] + b0['intercept']
    if above.any():
        bL = pw['bins'][-1]
        out[above] = pred[above] + bL['slope'] * pred[above] + bL['intercept']
    return out


# Phase 7R.3: 4-track bias correction A/B test with train and test RMSE

def metrics(y_true, y_pred):
    return {
        'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'mae':  float(mean_absolute_error(y_true, y_pred)),
        'r2':   float(r2_score(y_true, y_pred)),
    }


rows = []
piecewise_params = {}

for tgt, y_tr, y_te, tr_raw, te_raw, oof_pred in [
    ('T_C',    y_train_T, y_test_T, train_pred_T_raw, test_pred_T_raw, oof_T),
    ('P_kbar', y_train_P, y_test_P, train_pred_P_raw, test_pred_P_raw, oof_P),
]:
    # Track 1: Raw RF (no correction)
    m_tr = metrics(y_tr, tr_raw); m_te = metrics(y_te, te_raw)
    rows.append({'target': tgt, 'track': '1_raw_rf',
                 'rmse_train': m_tr['rmse'], 'mae_train': m_tr['mae'],
                 'rmse_test': m_te['rmse'], 'mae_test': m_te['mae'],
                 'r2_test': m_te['r2']})

    # Track 2: Global linear correction fit on TRAIN predictions (LEAK; for A/B comparison)
    lr_leak = LinearRegression().fit(tr_raw.reshape(-1, 1), y_tr - tr_raw)
    tr_corr_leak = tr_raw + lr_leak.predict(tr_raw.reshape(-1, 1))
    te_corr_leak = te_raw + lr_leak.predict(te_raw.reshape(-1, 1))
    m_tr = metrics(y_tr, tr_corr_leak); m_te = metrics(y_te, te_corr_leak)
    rows.append({'target': tgt, 'track': '2_linear_trainfit_leak',
                 'rmse_train': m_tr['rmse'], 'mae_train': m_tr['mae'],
                 'rmse_test': m_te['rmse'], 'mae_test': m_te['mae'],
                 'r2_test': m_te['r2']})

    # Track 3: Global linear correction fit on OOF predictions (honest)
    lr_oof = LinearRegression().fit(oof_pred.reshape(-1, 1), y_tr - oof_pred)
    tr_corr_oof = tr_raw + lr_oof.predict(tr_raw.reshape(-1, 1))
    te_corr_oof = te_raw + lr_oof.predict(te_raw.reshape(-1, 1))
    m_tr = metrics(y_tr, tr_corr_oof); m_te = metrics(y_te, te_corr_oof)
    rows.append({'target': tgt, 'track': '3_linear_ooffit',
                 'rmse_train': m_tr['rmse'], 'mae_train': m_tr['mae'],
                 'rmse_test': m_te['rmse'], 'mae_test': m_te['mae'],
                 'r2_test': m_te['r2']})

    # Track 4: Piecewise correction fit on OOF predictions
    pw = fit_piecewise_correction(oof_pred, y_tr, n_bins=5)
    piecewise_params[tgt] = pw
    tr_corr_pw = apply_piecewise_correction(tr_raw, pw)
    te_corr_pw = apply_piecewise_correction(te_raw, pw)
    m_tr = metrics(y_tr, tr_corr_pw); m_te = metrics(y_te, te_corr_pw)
    rows.append({'target': tgt, 'track': '4_piecewise_ooffit',
                 'rmse_train': m_tr['rmse'], 'mae_train': m_tr['mae'],
                 'rmse_test': m_te['rmse'], 'mae_test': m_te['mae'],
                 'r2_test': m_te['r2']})

ab_df = pd.DataFrame(rows)
ab_df.to_csv(RESULTS / 'nb07_ab_test_report.csv', index=False)
with open(RESULTS / 'nb07_piecewise_params.json', 'w') as f:
    json.dump(piecewise_params, f, indent=2)
print(ab_df.round(3).to_string(index=False))

target                  track  rmse_train  mae_train  rmse_test  mae_test  r2_test
   T_C               1_raw_rf       0.000      0.000     84.300    57.428    0.631
   T_C 2_linear_trainfit_leak       0.000      0.000     84.300    57.428    0.631
   T_C        3_linear_ooffit       5.235      4.062     84.504    57.714    0.629
   T_C     4_piecewise_ooffit      12.693     10.707     84.665    57.972    0.627
P_kbar               1_raw_rf       1.333      0.394      6.809     4.623    0.501
P_kbar 2_linear_trainfit_leak       1.234      0.577      6.729     4.568    0.513
P_kbar        3_linear_ooffit       1.653      1.146      6.596     4.478    0.532
P_kbar     4_piecewise_ooffit       1.784      1.275      6.636     4.699    0.526


## Phase 7R.3b: Bias-correction null-result check (v5)

The naive read of the A/B table above is "piecewise OOF correction reduced P RMSE
by roughly 0.2 kbar, so it should be the headline result." That is a single point
estimate. The question is whether the improvement survives sampling uncertainty.

We run a paired bootstrap (B=2000, resampling test indices with replacement) on
delta-RMSE between track 1 (raw RF) and track 4 (piecewise OOF-fit). Pairing is
essential: both tracks are evaluated on the same resampled test samples, so the
difference cancels sample-level variance that would otherwise dominate.

If the 95% CI of delta-RMSE contains zero, piecewise bias correction is a **null
result** - not a headline finding. The manuscript should then frame it as a
supplementary ablation that confirms the raw RF is already close to the information
ceiling on this feature set, rather than as a gain worth advertising.

In [6]:
# Phase 7R.3b (v5): paired bootstrap Delta-RMSE: raw RF vs piecewise OOF-fit
# Tests whether the ~0.2 kbar P improvement shown in the A/B table is a real
# gain or within sampling noise (null result).

def paired_bootstrap_delta_rmse(y_true, pred_a, pred_b, B=2000, seed=SEED_MODEL):
    """Δ = RMSE(a) - RMSE(b). Positive Δ favors b (smaller error).
    Pairing: same bootstrap indices used for both models."""
    rng = np.random.default_rng(seed)
    n = len(y_true)
    deltas = np.empty(B, dtype=float)
    for i in range(B):
        idx = rng.integers(0, n, size=n)
        y_b = y_true[idx]
        pa  = pred_a[idx]
        pb  = pred_b[idx]
        rmse_a = np.sqrt(np.mean((y_b - pa) ** 2))
        rmse_b = np.sqrt(np.mean((y_b - pb) ** 2))
        deltas[i] = rmse_a - rmse_b
    return deltas


# Piecewise-corrected test predictions (reuse stored params from cell above)
te_pw_T = apply_piecewise_correction(test_pred_T_raw, piecewise_params['T_C'])
te_pw_P = apply_piecewise_correction(test_pred_P_raw, piecewise_params['P_kbar'])

null_rows = []
for tgt, y_te, raw, pw in [
    ('T_C',    y_test_T, test_pred_T_raw, te_pw_T),
    ('P_kbar', y_test_P, test_pred_P_raw, te_pw_P),
]:
    rmse_raw = float(np.sqrt(mean_squared_error(y_te, raw)))
    rmse_pw  = float(np.sqrt(mean_squared_error(y_te, pw)))
    deltas = paired_bootstrap_delta_rmse(y_te, raw, pw, B=2000, seed=SEED_MODEL)
    lo, hi = np.quantile(deltas, [0.025, 0.975])
    median = float(np.median(deltas))
    ci_contains_zero = bool(lo <= 0.0 <= hi)
    null_rows.append({
        'target': tgt,
        'rmse_raw_rf': rmse_raw,
        'rmse_piecewise': rmse_pw,
        'delta_rmse_point': rmse_raw - rmse_pw,
        'delta_rmse_boot_median': median,
        'delta_rmse_ci_lo': float(lo),
        'delta_rmse_ci_hi': float(hi),
        'ci_contains_zero': ci_contains_zero,
        'interpretation': 'NULL: piecewise correction within sampling noise'
                          if ci_contains_zero else
                          'SIGNIFICANT: piecewise correction improves over raw RF',
    })

null_df = pd.DataFrame(null_rows)
null_df.to_csv(RESULTS / 'nb07_bias_correction_null_result.csv', index=False)

print('Paired bootstrap delta-RMSE (raw RF -> piecewise OOF-fit) on test set, B=2000:')
print(null_df.round(4).to_string(index=False))
print()
for _, r in null_df.iterrows():
    units = 'C' if r['target'] == 'T_C' else 'kbar'
    tag = '[NULL]' if r['ci_contains_zero'] else '[SIGNIFICANT]'
    print(f"{tag} {r['target']}: delta-RMSE point = {r['delta_rmse_point']:+.3f} {units}, "
          f"95% CI [{r['delta_rmse_ci_lo']:+.3f}, {r['delta_rmse_ci_hi']:+.3f}] {units}")


Paired bootstrap delta-RMSE (raw RF -> piecewise OOF-fit) on test set, B=2000:
target  rmse_raw_rf  rmse_piecewise  delta_rmse_point  delta_rmse_boot_median  delta_rmse_ci_lo  delta_rmse_ci_hi  ci_contains_zero                                   interpretation
   T_C      84.2999         84.6649           -0.3650                 -0.3590           -2.4608            1.7439              True NULL: piecewise correction within sampling noise
P_kbar       6.8089          6.6355            0.1733                  0.1721           -0.0411            0.3873              True NULL: piecewise correction within sampling noise

[NULL] T_C: delta-RMSE point = -0.365 C, 95% CI [-2.461, +1.744] C
[NULL] P_kbar: delta-RMSE point = +0.173 kbar, 95% CI [-0.041, +0.387] kbar


In [7]:
# Phase 7R.4: QRF leakage A/B test - train-set coverage vs OOF coverage
# Track A (LEAK): QRF fit on full training set, coverage measured on training set.
# Track B (honest): 10-fold StratifiedGroupKFold OOF coverage on training set.
# Target nominal coverage for the 16/84 quantile band is ~68%.

def coverage(lo, hi, y):
    return float(np.mean((y >= lo) & (y <= hi)))


qrf_T_full = RandomForestQuantileRegressor(**params_T, random_state=SEED_MODEL, n_jobs=-1)
qrf_T_full.fit(X_train, y_train_T)
qrf_P_full = RandomForestQuantileRegressor(**params_P, random_state=SEED_MODEL, n_jobs=-1)
qrf_P_full.fit(X_train, y_train_P)

# Track A (LEAK): evaluate the full-fit QRF on the training set it was fit on
q_T_tr = qrf_T_full.predict(X_train, quantiles=[0.16, 0.5, 0.84])
q_P_tr = qrf_P_full.predict(X_train, quantiles=[0.16, 0.5, 0.84])
cov_T_leak = coverage(q_T_tr[:, 0], q_T_tr[:, 2], y_train_T)
cov_P_leak = coverage(q_P_tr[:, 0], q_P_tr[:, 2], y_train_P)

# Track B (honest): 10-fold OOF coverage
oof_T_lo, oof_T_md, oof_T_hi = oof_qrf(X_train, y_train_T, groups_train, params_T, SEED_MODEL)
oof_P_lo, oof_P_md, oof_P_hi = oof_qrf(X_train, y_train_P, groups_train, params_P, SEED_MODEL)
cov_T_oof = coverage(oof_T_lo, oof_T_hi, y_train_T)
cov_P_oof = coverage(oof_P_lo, oof_P_hi, y_train_P)

# Test-set coverage using the full-fit QRF models (reported alongside for reference)
q_T_te = qrf_T_full.predict(X_test, quantiles=[0.16, 0.5, 0.84])
q_P_te = qrf_P_full.predict(X_test, quantiles=[0.16, 0.5, 0.84])
cov_T_test = coverage(q_T_te[:, 0], q_T_te[:, 2], y_test_T)
cov_P_test = coverage(q_P_te[:, 0], q_P_te[:, 2], y_test_P)

ab_cov = pd.DataFrame([
    {'target': 'T_C',    'track': 'A_trainfit_leak', 'coverage_68': cov_T_leak,
     'rmse_median': float(np.sqrt(mean_squared_error(y_train_T, q_T_tr[:, 1])))},
    {'target': 'T_C',    'track': 'B_oof_honest',    'coverage_68': cov_T_oof,
     'rmse_median': float(np.sqrt(mean_squared_error(y_train_T, oof_T_md)))},
    {'target': 'T_C',    'track': 'C_test',          'coverage_68': cov_T_test,
     'rmse_median': float(np.sqrt(mean_squared_error(y_test_T, q_T_te[:, 1])))},
    {'target': 'P_kbar', 'track': 'A_trainfit_leak', 'coverage_68': cov_P_leak,
     'rmse_median': float(np.sqrt(mean_squared_error(y_train_P, q_P_tr[:, 1])))},
    {'target': 'P_kbar', 'track': 'B_oof_honest',    'coverage_68': cov_P_oof,
     'rmse_median': float(np.sqrt(mean_squared_error(y_train_P, oof_P_md)))},
    {'target': 'P_kbar', 'track': 'C_test',          'coverage_68': cov_P_test,
     'rmse_median': float(np.sqrt(mean_squared_error(y_test_P, q_P_te[:, 1])))},
])
ab_cov.to_csv(RESULTS / 'nb07_qrf_ab_coverage.csv', index=False)

# Save canonical QRF models for downstream use (Fig 11, Phase 11, manuscript)
joblib.dump(qrf_T_full, MODELS / 'model_QRF_T_C_opx_liq.joblib')
joblib.dump(qrf_P_full, MODELS / 'model_QRF_P_kbar_opx_liq.joblib')
print(ab_cov.round(3).to_string(index=False))

target           track  coverage_68  rmse_median
   T_C A_trainfit_leak        1.000        0.000
   T_C    B_oof_honest        0.737       73.373
   T_C          C_test        0.707       84.300
P_kbar A_trainfit_leak        1.000        1.584
P_kbar    B_oof_honest        0.831        5.604
P_kbar          C_test        0.615        6.935


## Phase 7R.5: Split-conformal calibration (v5, multi-alpha primary)

Split-conformal calibration supplies a distribution-free coverage guarantee: fit the
base model on 90% of the training data, evaluate absolute residuals on the remaining
10% calibration split, and take the finite-sample-corrected (1 - alpha) quantile of
those residuals. Symmetric intervals of that half-width around the median prediction
cover the true target with probability at least 1 - alpha under exchangeability
(Vovk; Lei et al. 2018).

The v5 spec promotes split-conformal to the primary uncertainty estimate for NB08 /
NB10 / manuscript, with multi-alpha (68 / 80 / 90 / 95%) intervals reported so
readers can pick the coverage level appropriate for their use case. The previous
QRF-based 16-84 band (nominal 68%) is retained but demoted to supplementary because
its empirical coverage (61-71%) missed the nominal level and varied by target.

Primary intervals for the manuscript body: P +/- q_hat_P_90, T +/- q_hat_T_90 (90%
nominal coverage).


In [8]:
# Phase 7R.5 (v5): Multi-alpha split-conformal calibration.
# Fit base QRF on 90%, calibrate residuals on 10%, report empirical vs nominal
# coverage at alpha in {0.32, 0.20, 0.10, 0.05} -> nominal 68, 80, 90, 95%.
from src.calibration import conformal_quantile, conformal_intervals, compute_coverage
from sklearn.model_selection import train_test_split
from sklearn.base import clone
import json

ALPHAS = [0.32, 0.20, 0.10, 0.05]
NOMINAL = {a: 1.0 - a for a in ALPHAS}

X_fit, X_cal, y_fit_P, y_cal_P = train_test_split(
    X_train, y_train_P, test_size=0.10, random_state=SEED_SPLIT
)
_, _, y_fit_T, y_cal_T = train_test_split(
    X_train, y_train_T, test_size=0.10, random_state=SEED_SPLIT
)

qrf_P_cal = clone(qrf_P_full).fit(X_fit, y_fit_P)
qrf_T_cal = clone(qrf_T_full).fit(X_fit, y_fit_T)

pred_P_cal = predict_median(qrf_P_cal, X_cal)
pred_T_cal = predict_median(qrf_T_cal, X_cal)
resid_P = y_cal_P - pred_P_cal
resid_T = y_cal_T - pred_T_cal

pred_P_test = predict_median(qrf_P_full, X_test)
pred_T_test = predict_median(qrf_T_full, X_test)

conformal_rows = []
qhat_store = {'n_calibration': int(len(y_cal_P)), 'alphas': {}}
for alpha in ALPHAS:
    q_P = conformal_quantile(resid_P, alpha=alpha)
    q_T = conformal_quantile(resid_T, alpha=alpha)
    P_lo, P_hi = conformal_intervals(pred_P_test, q_P)
    T_lo, T_hi = conformal_intervals(pred_T_test, q_T)
    cov_P = compute_coverage(y_test_P, P_lo, P_hi)
    cov_T = compute_coverage(y_test_T, T_lo, T_hi)
    conformal_rows.extend([
        {'target': 'P_kbar', 'alpha': alpha, 'nominal_coverage': NOMINAL[alpha],
         'q_hat': float(q_P), 'empirical_coverage': float(cov_P)},
        {'target': 'T_C',    'alpha': alpha, 'nominal_coverage': NOMINAL[alpha],
         'q_hat': float(q_T), 'empirical_coverage': float(cov_T)},
    ])
    qhat_store['alphas'][f'{alpha:.2f}'] = {
        'nominal_coverage':  NOMINAL[alpha],
        'q_hat_P_kbar':      float(q_P),
        'q_hat_T_C':         float(q_T),
        'empirical_cov_P':   float(cov_P),
        'empirical_cov_T':   float(cov_T),
    }

conformal_df = pd.DataFrame(conformal_rows)
conformal_df.to_csv(RESULTS / 'nb07_conformal_multi_alpha.csv', index=False)
print('Multi-alpha split-conformal calibration:')
print(conformal_df.round(4).to_string(index=False))

# Primary manuscript intervals: 90% nominal.
q_hat_P = qhat_store['alphas']['0.10']['q_hat_P_kbar']
q_hat_T = qhat_store['alphas']['0.10']['q_hat_T_C']
cov_P   = qhat_store['alphas']['0.10']['empirical_cov_P']
cov_T   = qhat_store['alphas']['0.10']['empirical_cov_T']
print(f'\nPrimary manuscript intervals (nominal 90%):')
print(f'  q_hat_P_90 = {q_hat_P:.2f} kbar  empirical {cov_P:.1%}')
print(f'  q_hat_T_90 = {q_hat_T:.1f} C     empirical {cov_T:.1%}')

# Back-compat: keep the legacy single-alpha JSON so older downstream code
# (NB08 / NBF) continues to run. Now populated from alpha=0.10.
qhat_store['primary_alpha']    = 0.10
qhat_store['q_hat_P_kbar']     = q_hat_P
qhat_store['q_hat_T_C']        = q_hat_T
qhat_store['coverage_P']       = cov_P
qhat_store['coverage_T']       = cov_T
qhat_store['alpha']            = 0.10
with open(RESULTS / 'nb07_conformal_qhat.json', 'w') as f:
    json.dump(qhat_store, f, indent=2)


Multi-alpha split-conformal calibration:
target  alpha  nominal_coverage    q_hat  empirical_coverage
P_kbar   0.32              0.68   1.7034              0.3678
   T_C   0.32              0.68  40.0000              0.5402
P_kbar   0.20              0.80   5.0000              0.6552
   T_C   0.20              0.80  50.0000              0.6207
P_kbar   0.10              0.90  10.0000              0.9195
   T_C   0.10              0.90  90.0000              0.7931
P_kbar   0.05              0.95  13.7083              0.9713
   T_C   0.05              0.95 170.0000              0.9540

Primary manuscript intervals (nominal 90%):
  q_hat_P_90 = 10.00 kbar  empirical 92.0%
  q_hat_T_90 = 90.0 C     empirical 79.3%


In [9]:
# Persist per-sample test predictions (QRF + piecewise-corrected RF) for Fig 11
pw_T = piecewise_params['T_C']
pw_P = piecewise_params['P_kbar']
te_pw_T = apply_piecewise_correction(test_pred_T_raw, pw_T)
te_pw_P = apply_piecewise_correction(test_pred_P_raw, pw_P)

preds_out = pd.DataFrame({
    'index':        idx_te,
    'T_true':       y_test_T,
    'T_rf_raw':     test_pred_T_raw,
    'T_rf_pwcorr':  te_pw_T,
    'T_qrf_median': q_T_te[:, 1],
    'T_qrf_lo':     q_T_te[:, 0],
    'T_qrf_hi':     q_T_te[:, 2],
    'P_true':       y_test_P,
    'P_rf_raw':     test_pred_P_raw,
    'P_rf_pwcorr':  te_pw_P,
    'P_qrf_median': q_P_te[:, 1],
    'P_qrf_lo':     q_P_te[:, 0],
    'P_qrf_hi':     q_P_te[:, 2],
})
preds_out.to_csv(RESULTS / 'nb07_test_predictions.csv', index=False)
print(f'Saved test predictions n={len(preds_out)} to nb07_test_predictions.csv')

Saved test predictions n=174 to nb07_test_predictions.csv


In [10]:
# Verification assertions
assert (RESULTS / 'nb07_ab_test_report.csv').exists()
assert (RESULTS / 'nb07_piecewise_params.json').exists()
assert (RESULTS / 'nb07_qrf_ab_coverage.csv').exists()
assert (RESULTS / 'nb07_test_predictions.csv').exists()
assert (RESULTS / 'nb07_conformal_multi_alpha.csv').exists()
assert (RESULTS / 'nb07_conformal_qhat.json').exists()
assert (RESULTS / 'nb07_bias_correction_null_result.csv').exists()
assert (MODELS / 'model_QRF_T_C_opx_liq.joblib').exists()
assert (MODELS / 'model_QRF_P_kbar_opx_liq.joblib').exists()

ab_check = pd.read_csv(RESULTS / 'nb07_ab_test_report.csv')
assert set(ab_check['track']) == {'1_raw_rf', '2_linear_trainfit_leak',
                                  '3_linear_ooffit', '4_piecewise_ooffit'}
assert set(ab_check['target']) == {'T_C', 'P_kbar'}
assert {'rmse_train', 'rmse_test', 'r2_test'}.issubset(ab_check.columns)

cov_check = pd.read_csv(RESULTS / 'nb07_qrf_ab_coverage.csv')
assert set(cov_check['track']) == {'A_trainfit_leak', 'B_oof_honest', 'C_test'}

conf_check = pd.read_csv(RESULTS / 'nb07_conformal_multi_alpha.csv')
assert set(conf_check['alpha']) == {0.32, 0.20, 0.10, 0.05}
assert set(conf_check['target']) == {'T_C', 'P_kbar'}

null_check = pd.read_csv(RESULTS / 'nb07_bias_correction_null_result.csv')
assert set(null_check['target']) == {'T_C', 'P_kbar'}
assert {'delta_rmse_ci_lo', 'delta_rmse_ci_hi', 'ci_contains_zero'}.issubset(null_check.columns)

print('=== PHASE 7R COMPLETE ===')
print(f'  winning feature set: {WIN_FEAT}')
print(f'  AB test tracks: {sorted(ab_check["track"].unique().tolist())}')
print(f'  QRF coverage tracks: {sorted(cov_check["track"].unique().tolist())}')
print(f'  conformal alphas: {sorted(conf_check["alpha"].unique().tolist())}')
print(f'  bias-correction null test: saved')

=== PHASE 7R COMPLETE ===
  winning feature set: pwlr
  AB test tracks: ['1_raw_rf', '2_linear_trainfit_leak', '3_linear_ooffit', '4_piecewise_ooffit']
  QRF coverage tracks: ['A_trainfit_leak', 'B_oof_honest', 'C_test']
  conformal alphas: [0.05, 0.1, 0.2, 0.32]
  bias-correction null test: saved
